In [1]:
import os
from collections import Counter

raw = "data/photos/raw"
IMG = (".jpg", ".jpeg", ".png")

# 1) roboflow_quality: photos per class per split
print("== roboflow_quality ==")
for split in ["train", "valid", "test"]:
    p = f"{raw}/roboflow_quality/{split}"
    if os.path.isdir(p):
        print(split, {c: len([f for f in os.listdir(f"{p}/{c}") if f.lower().endswith(IMG)])
                      for c in sorted(os.listdir(p)) if os.path.isdir(f"{p}/{c}")})

# 2) roboflow_turbidity: class names + how many label boxes per class
print("\n== roboflow_turbidity ==")
print(open(f"{raw}/roboflow_turbidity/data.yaml").read())
cnt = Counter()
for split in ["train", "valid", "test"]:
    ld = f"{raw}/roboflow_turbidity/{split}/labels"
    if os.path.isdir(ld):
        for f in os.listdir(ld):
            for line in open(f"{ld}/{f}"):
                if line.strip():
                    cnt[line.split()[0]] += 1
print("boxes per class id:", dict(cnt))

# 3) mendeley
print("\n== mendeley_polluted ==")
for sub in ["clean", "images", "masks"]:
    p = f"{raw}/mendeley_polluted/{sub}"
    print(sub, len([f for f in os.listdir(p) if f.lower().endswith(IMG)]))

== roboflow_quality ==
train {'Clear': 25, 'Foam': 354, 'Highly_Turbid': 90, 'Medium_Turbid': 148, 'Unlabeled': 8}
valid {'Clear': 4, 'Foam': 88, 'Highly_Turbid': 27, 'Medium_Turbid': 40, 'Unlabeled': 2}
test {'Clear': 1, 'Foam': 263, 'Highly_Turbid': 17, 'Medium_Turbid': 96, 'Unlabeled': 1}

== roboflow_turbidity ==
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['clean', 'turbid']

roboflow:
  workspace: dataregenerator
  project: turbidity
  version: 4
  license: CC BY 4.0
  url: https://universe.roboflow.com/dataregenerator/turbidity/dataset/4
boxes per class id: {'1': 2194, '0': 374}

== mendeley_polluted ==
clean 300
images 1236
masks 1236


In [3]:
import os, shutil, random, re, yaml
from collections import Counter, defaultdict

RAW = "data/photos/raw"
OUT = "data/photos/merged"
IMG = (".jpg", ".jpeg", ".png")
random.seed(42)

if os.path.exists(OUT):
    shutil.rmtree(OUT)

def put(src, split, cls, prefix):
    d = f"{OUT}/{split}/{cls}"
    os.makedirs(d, exist_ok=True)
    shutil.copy(src, f"{d}/{prefix}_{os.path.basename(src)}")

# 1) roboflow_quality: keep its splits, test goes into valid
q_map = {"Clear": "Clean", "Medium_Turbid": "Cloudy",
         "Highly_Turbid": "Cloudy", "Foam": "Foam"}
for s_in, s_out in [("train", "train"), ("valid", "valid"), ("test", "valid")]:
    for folder, cls in q_map.items():
        d = f"{RAW}/roboflow_quality/{s_in}/{folder}"
        if os.path.isdir(d):
            for f in os.listdir(d):
                if f.lower().endswith(IMG):
                    put(f"{d}/{f}", s_out, cls, "rfq")

# 2) roboflow_turbidity: class = most common box label in the photo
names = yaml.safe_load(open(f"{RAW}/roboflow_turbidity/data.yaml"))["names"]
t_map = {"clean": "Clean", "turbid": "Cloudy"}
for s_in, s_out in [("train", "train"), ("valid", "valid"), ("test", "valid")]:
    img_dir = f"{RAW}/roboflow_turbidity/{s_in}/images"
    lbl_dir = f"{RAW}/roboflow_turbidity/{s_in}/labels"
    if not os.path.isdir(img_dir):
        continue
    for f in os.listdir(img_dir):
        lbl = f"{lbl_dir}/{os.path.splitext(f)[0]}.txt"
        if not os.path.exists(lbl):
            continue
        ids = [l.split()[0] for l in open(lbl) if l.strip()]
        if not ids:
            continue
        cls = t_map[names[int(Counter(ids).most_common(1)[0][0])]]
        put(f"{img_dir}/{f}", s_out, cls, "rft")

# 3) mendeley: clean = split by file, polluted = split by original photo
M = f"{RAW}/mendeley_polluted"
clean = sorted(f for f in os.listdir(f"{M}/clean") if f.lower().endswith(IMG))
random.shuffle(clean)
cut = int(len(clean) * 0.8)
for i, f in enumerate(clean):
    put(f"{M}/clean/{f}", "train" if i < cut else "valid", "Clean", "mdl")

groups = defaultdict(list)
for f in os.listdir(f"{M}/images"):
    if f.lower().endswith(IMG):
        groups[re.sub(r"_aug\d+", "", os.path.splitext(f)[0])].append(f)
keys = sorted(groups)
random.shuffle(keys)
cut = int(len(keys) * 0.8)
for i, k in enumerate(keys):
    for f in groups[k]:
        put(f"{M}/images/{f}", "train" if i < cut else "valid", "Polluted", "mdl")

# result
for split in ["train", "valid"]:
    print(split, {c: len(os.listdir(f"{OUT}/{split}/{c}"))
                  for c in sorted(os.listdir(f"{OUT}/{split}"))})

train {'Clean': 593, 'Cloudy': 2124, 'Foam': 354, 'Polluted': 984}
valid {'Clean': 111, 'Cloudy': 485, 'Foam': 351, 'Polluted': 252}
